In [50]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import utils
import textwrap
import tensorflow as tf 
wrapper = textwrap.TextWrapper(width=70)

tf.keras.utils.set_random_seed(10)

In [4]:
data_dir = 'data/corpus'

In [35]:
train_data, test_data = utils.get_train_test_data(data_dir)
print(len(train_data),len(test_data))

14732 819


In [36]:
#preprocessing data
document, summary = utils.preprocess(train_data)
document_test, summary_test = utils.preprocess(test_data)

In [37]:
# Getting the emboddings of the sequences
filters = '!"#$%&()*+,-./:;<=>?@\\^_`{|}~\t\n'
oov_token = '[UNK]'
tokonizer =  tf.keras.preprocessing.text.Tokenizer(filters=filters,oov_token=oov_token,lower=False)
document_and_summary = pd.concat([document,summary],ignore_index=True) #CONCATING IN VETICAL FORM'
tokonizer.fit_on_texts(document_and_summary)
inputs = tokonizer.texts_to_sequences(document)
target = tokonizer.texts_to_sequences(document)
vocab_size = len(tokonizer.word_index)+1
print(f'Size of vocabulary: {vocab_size}')

Size of vocabulary: 34250


In [54]:
#padding the sequence
input_maxlen = 150
output_maxlen = 50

inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=input_maxlen, padding='post', truncating='post')
target = tf.keras.preprocessing.sequence.pad_sequences(target, maxlen=output_maxlen, padding='post', truncating='post')

inputs = tf.cast(inputs, dtype=tf.float32)
target = tf.cast(target, dtype=tf.float32)

In [55]:
#making dataset

In [56]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
dataset = tf.data.Dataset.from_tensor_slices((inputs,target)).shuffle(BATCH_SIZE).batch(BATCH_SIZE)

In [103]:
#Positional Encoding
def positional_encoding(positions, d_model):
    # positions (int): Maximum number of positions to be encoded 
    # d_model (int): Encoding size
    positions = np.arange(positions)[:,np.newaxis]
    k = np.arange(d_model)[np.newaxis,:]
    i = k//2
    
    angle_rates = 1 / np.power(10000, (2 * i) / np.float32(d_model))
    angle_rads = positions * angle_rates
    angle_rads[:,0::2] = np.sin(angle_rads[:,0::2])
    angle_rads[:,1::2] = np.cos(angle_rads[:,1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return tf.cast(pos_encoding, dtype=tf.float32)

In [108]:
# creating padding mask to know the actual and paded data 
def create_padding(decoder_tokens_ids):
    ids =  tf.cast(tf.math.equal(decoder_tokens_ids,0), tf.float32)
    return ids[:,tf.newaxis,...]

In [109]:
create_padding(target)

<tf.Tensor: shape=(14732, 1, 50), dtype=float32, numpy=
array([[[0., 0., 0., ..., 1., 1., 1.]],

       [[0., 0., 0., ..., 1., 1., 1.]],

       [[0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.]]], dtype=float32)>